<a href="https://colab.research.google.com/github/mohamedelguendy/Secure-Token-Based-Authentication-Framework/blob/main/Secure_Token_Based_Authentication_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
pip install pwinput -q

In [7]:
import hashlib
import os
import jwt
import datetime
import hmac
import getpass
import sys
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes

# =========================
# SMART PASSWORD INPUT (NO DOUBLE PROMPT)
# =========================

def secure_input(prompt="Password: "):
    if sys.stdin.isatty():
        try:
            import pwinput
            return pwinput.pwinput(prompt=prompt, mask="*")
        except:
            return getpass.getpass(prompt)
    else:
        return getpass.getpass(prompt)

# =========================
# DATABASE + SECRET
# =========================

users_db = {}
SECRET_KEY = "mysecretkey"

# =========================
# DIGITAL SIGNATURE KEYS
# =========================

private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_key = private_key.public_key()

# =========================
# FUNCTIONS
# =========================

def register():
    username = input("Enter username: ")
    password = secure_input("Enter password: ")
    role = input("Enter role (admin/user): ")

    salt = os.urandom(16)
    password_hash = hashlib.sha256(salt + password.encode()).hexdigest()

    users_db[username] = {
        "salt": salt,
        "password_hash": password_hash,
        "role": role
    }

    print("User registered successfully\n")


def login():
    username = input("Username: ")
    password = secure_input("Password: ")

    user = users_db.get(username)
    if not user:
        print("User not found\n")
        return None

    new_hash = hashlib.sha256(user["salt"] + password.encode()).hexdigest()

    if new_hash == user["password_hash"]:
        print("Login successful\n")
        return username
    else:
        print("Invalid credentials\n")
        return None

# =========================
# Forgot Password (Reset)
# =========================

def forgot_password():
    username = input("Enter your username: ")
    user = users_db.get(username)

    if not user:
        print("User not found\n")
        return

    role = input("Enter your role for verification: ")

    if role != user["role"]:
        print("Verification failed\n")
        return

    new_password = secure_input("Enter NEW password: ")

    salt = os.urandom(16)
    password_hash = hashlib.sha256(salt + new_password.encode()).hexdigest()

    users_db[username]["salt"] = salt
    users_db[username]["password_hash"] = password_hash

    print("Password reset successful\n")


def generate_token(username):
    payload = {
        "username": username,
        "role": users_db[username]["role"],
        "exp": datetime.datetime.utcnow() + datetime.timedelta(seconds=20)
    }
    return jwt.encode(payload, SECRET_KEY, algorithm="HS256")


def verify_token(token):
    try:
        decoded = jwt.decode(token, SECRET_KEY, algorithms=["HS256"])
        print("Token valid:", decoded)
        return decoded
    except jwt.ExpiredSignatureError:
        print("Token expired")
    except jwt.InvalidTokenError:
        print("Invalid/tampered token")


def generate_hmac(message):
    return hmac.new(SECRET_KEY.encode(), message.encode(), hashlib.sha256).hexdigest()


def verify_hmac(message, mac):
    return generate_hmac(message) == mac


def sign_message(message):
    return private_key.sign(
        message.encode(),
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )


def verify_signature(message, signature):
    try:
        public_key.verify(
            signature,
            message.encode(),
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH
            ),
            hashes.SHA256()
        )
        print("Signature valid")
    except:
        print("Signature invalid")


def access_control(decoded):
    if decoded["role"] == "admin":
        print("Full access granted")
    else:
        print("Limited access")


# =========================
# MAIN PROGRAM
# =========================

while True:
    print("1. Register")
    print("2. Login")
    print("3. Forgot Password")
    print("4. Exit")

    choice = input("Choose option: ")

    if choice == "1":
        register()

    elif choice == "2":
        user = login()
        if user:
            token = generate_token(user)
            print("Your JWT:", token, "\n")

            decoded = verify_token(token)

            if decoded:
                access_control(decoded)

                msg = input("\nEnter a message: ")

                mac = generate_hmac(msg)
                print("Generated HMAC:", mac)
                print("HMAC valid:", verify_hmac(msg, mac))

                sig = sign_message(msg)
                verify_signature(msg, sig)

                print("\n--- Tampering Test ---")
                fake_msg = msg + " hacked"
                print("Tampered HMAC:", verify_hmac(fake_msg, mac))
                verify_signature(fake_msg, sig)

                print("\n--- Token Tampering ---")
                verify_token(token + "abc")

                import time
                print("\nWaiting for token to expire...")
                time.sleep(21)
                verify_token(token)

    elif choice == "3":
        forgot_password()

    elif choice == "4":
        break

    else:
        print("Invalid choice\n")

1. Register
2. Login
3. Forgot Password
4. Exit
Choose option: 1
Enter username: sd
Enter password: ··········
Enter role (admin/user): admin
User registered successfully

1. Register
2. Login
3. Forgot Password
4. Exit
Choose option: 3
Enter your username: sd
Enter your role for verification: admin
Enter NEW password: ··········
Password reset successful

1. Register
2. Login
3. Forgot Password
4. Exit
Choose option: 4
